<a href="https://colab.research.google.com/github/moshuhuang/RouteGuard-last-mile-address-risk/blob/main/02_feature_engineering_and_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Fix widget metadata for GitHub rendering
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Git Projects/address_risk_dataset_30k.csv')
print(df.shape)
print(df['is_address_risk'].value_counts())
print(df.head(3))

In [ ]:
import re

def engineer_features(df):
    addr = df['input_address'].fillna('')

    # 长度类
    df['address_length'] = addr.str.len()
    df['word_count'] = addr.str.split().str.len()
    df['digit_count'] = addr.str.count(r'\d')
    df['comma_count'] = addr.str.count(',')
    df['special_char_count'] = addr.str.count(r'[^a-zA-Z0-9\s,]')

    # 结构信号
    df['has_house_number'] = addr.str.match(r'^\d+').astype(int)
    df['has_unit'] = addr.str.contains(
        r'\b(APT|UNIT|STE|SUITE|#|LOT|BLDG)\b', case=False).astype(int)
    df['has_zip'] = addr.str.contains(r'\b\d{5}\b').astype(int)
    df['has_city'] = addr.str.contains(r',\s*[A-Z]').astype(int)
    df['is_street_only'] = (
        (df['has_house_number'] == 0) &
        (df['has_zip'] == 0)
    ).astype(int)
    df['unit_keyword_no_number'] = (
        addr.str.contains(r'\b(APT|UNIT|STE|SUITE)\b', case=False) &
        ~addr.str.contains(r'\b(APT|UNIT|STE|SUITE)\s*\d+', case=False)
    ).astype(int)
    df['has_ambiguous_abbrev'] = addr.str.contains(
        r'\bWC\b|\bNC\b|\bEC\b|\bSC\b', case=False).astype(int)

    return df

df = engineer_features(df)

feature_cols = [
    'address_length', 'word_count', 'digit_count', 'comma_count',
    'special_char_count', 'has_house_number', 'has_unit', 'has_zip',
    'has_city', 'is_street_only', 'unit_keyword_no_number', 'has_ambiguous_abbrev'
]

print(df[feature_cols].describe())
print("\n缺失值:", df[feature_cols].isnull().sum().sum())

In [ ]:
from sklearn.model_selection import train_test_split

X = df[feature_cols]
y = df['is_address_risk']

# First split off test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# Split remaining into train (70%) and val (15%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")
print(f"Train risk rate: {y_train.mean():.3f}")
print(f"Val risk rate:   {y_val.mean():.3f}")
print(f"Test risk rate:  {y_test.mean():.3f}")

## Rule Based Model

In [ ]:
def rule_based_predict(df):
    risk = (
        (df['has_zip'] == 0) |
        (df['has_house_number'] == 0) |
        (df['is_street_only'] == 1) |
        (df['unit_keyword_no_number'] == 1)
    ).astype(int)
    return risk

from sklearn.metrics import classification_report, roc_auc_score

y_pred_rule = rule_based_predict(X_val)
print(classification_report(y_val, y_pred_rule))
print("AUC:", roc_auc_score(y_val, y_pred_rule))

## Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# Scale features - LR is sensitive to feature scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Train
lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)

# Evaluate
y_pred_lr = lr.predict(X_val_scaled)
y_prob_lr = lr.predict_proba(X_val_scaled)[:, 1]

print("=== Logistic Regression ===")
print(classification_report(y_val, y_pred_lr))
print("AUC:", roc_auc_score(y_val, y_prob_lr))

## XGBOOST

In [ ]:
from xgboost import XGBClassifier

# Calculate scale_pos_weight to handle class imbalance
# = number of negative samples / number of positive samples
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

# Train
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb.fit(X_train, y_train)

# Evaluate
y_pred_xgb = xgb.predict(X_val)
y_prob_xgb = xgb.predict_proba(X_val)[:, 1]

print("\n=== XGBoost (Structured Features) ===")
print(classification_report(y_val, y_pred_xgb))
print("AUC:", roc_auc_score(y_val, y_prob_xgb))

## Text Embedding

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for input_address
print("Generating embeddings...")
train_idx = X_train.index
val_idx = X_val.index
test_idx = X_test.index

train_embeddings = embedder.encode(df.loc[train_idx, 'input_address'].tolist(),
                                    batch_size=256, show_progress_bar=True)
val_embeddings = embedder.encode(df.loc[val_idx, 'input_address'].tolist(),
                                  batch_size=256, show_progress_bar=True)
test_embeddings = embedder.encode(df.loc[test_idx, 'input_address'].tolist(),
                                   batch_size=256, show_progress_bar=True)

print(f"Embedding shape: {train_embeddings.shape}")

## XGBoost (Structured + Embedding Fusion)

In [ ]:
# Combine structured features + embeddings
X_train_fused = np.hstack([X_train.values, train_embeddings])
X_val_fused = np.hstack([X_val.values, val_embeddings])
X_test_fused = np.hstack([X_test.values, test_embeddings])

print(f"Fused feature shape: {X_train_fused.shape}")

# Train
xgb_fused = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb_fused.fit(X_train_fused, y_train)

# Evaluate
y_pred_fused = xgb_fused.predict(X_val_fused)
y_prob_fused = xgb_fused.predict_proba(X_val_fused)[:, 1]

print("\n=== XGBoost (Structured + Embedding) ===")
print(classification_report(y_val, y_pred_fused))
print("AUC:", roc_auc_score(y_val, y_prob_fused))

## Threshold Optimization

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

# Find optimal threshold for F1 on fused model
precisions, recalls, thresholds = precision_recall_curve(y_val, y_prob_fused)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f"Default threshold (0.5) → Recall: 0.48, F1: 0.59")
print(f"Optimal threshold: {best_threshold:.3f}")

y_pred_optimal = (y_prob_fused >= best_threshold).astype(int)
print("\n=== XGBoost Fused (Optimal Threshold) ===")
print(classification_report(y_val, y_pred_optimal))

## SMOTE + XGBoost

In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_fused_sm, y_train_sm = sm.fit_resample(X_train_fused, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE: {pd.Series(y_train_sm).value_counts().to_dict()}")

xgb_smote = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb_smote.fit(X_train_fused_sm, y_train_sm)

y_prob_smote = xgb_smote.predict_proba(X_val_fused)[:, 1]
y_pred_smote = (y_prob_smote >= 0.4).astype(int)  # lower threshold

print("\n=== XGBoost Fused + SMOTE ===")
print(classification_report(y_val, y_pred_smote))
print("AUC:", roc_auc_score(y_val, y_prob_smote))

## Final Evaluation on Test Set

In [ ]:
# Final model: XGBoost Fused (best val F1 and AUC)
y_pred_test = xgb_fused.predict(X_test_fused)
y_prob_test = xgb_fused.predict_proba(X_test_fused)[:, 1]

print("=" * 50)
print("FINAL MODEL EVALUATION ON HELD-OUT TEST SET")
print("Model: XGBoost (Structured Features + Text Embedding)")
print("=" * 50)
print(classification_report(y_test, y_pred_test))
print("AUC:", roc_auc_score(y_test, y_prob_test))

# Summary table
print("\n=== Model Comparison Summary (Validation Set) ===")
results = {
    'Model': ['Rule-based', 'Logistic Regression',
              'XGBoost (Structured)', 'XGBoost (Fused)', 'XGBoost + SMOTE'],
    'Recall': [0.30, 0.44, 0.41, 0.48, 0.56],
    'F1':     [0.46, 0.52, 0.54, 0.59, 0.50],
    'AUC':    [0.65, 0.74, 0.73, 0.75, 0.73]
}
print(pd.DataFrame(results).to_string(index=False))
print("\nFinal model selected: XGBoost (Fused) — best F1 (0.59) and AUC (0.75)")

In [ ]:
import IPython.display as display

# Clear all widget state
display.clear_output()

# This clears widget metadata from notebook
get_ipython().kernel.comm_manager.comms = {}
print("Done. Now save to GitHub.")